In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# Print working directory and list files
print(f"Current working directory: {os.getcwd()}")
print(f"Files in directory: {os.listdir()}")

# Standard EU27 country mapping
country_mapping = {
    'Austria': 'Austria', 'Belgium': 'Belgium', 'Bulgaria': 'Bulgaria',
    'Croatia': 'Croatia', 'Cyprus': 'Cyprus', 'Czechia': 'Czech Republic',
    'Denmark': 'Denmark', 'Estonia': 'Estonia', 'Finland': 'Finland',
    'France': 'France', 'Germany': 'Germany', 'Greece': 'Greece',
    'Hungary': 'Hungary', 'Ireland': 'Ireland', 'Italy': 'Italy',
    'Latvia': 'Latvia', 'Lithuania': 'Lithuania', 'Luxembourg': 'Luxembourg',
    'Malta': 'Malta', 'Netherlands': 'Netherlands', 'Poland': 'Poland',
    'Portugal': 'Portugal', 'Romania': 'Romania', 'Slovakia': 'Slovakia',
    'Slovenia': 'Slovenia', 'Spain': 'Spain', 'Sweden': 'Sweden'
}

def clean_food_waste_data(file_path, population_df):
    print("🍽️ Cleaning Food Waste Dataset...")
    if not os.path.exists(file_path):
        print(f"Error: {file_path} not found. Please check the file path.")
        return pd.DataFrame()
    
    try:
        df = pd.read_csv(file_path)
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return pd.DataFrame()
    
    # Check required columns
    essential_cols = ['geo', 'TIME_PERIOD', 'nace_r2', 'unit', 'OBS_VALUE']
    missing_cols = [col for col in essential_cols if col not in df.columns]
    if missing_cols:
        print(f"Error: Missing columns in Waste.csv: {missing_cols}")
        return pd.DataFrame()
    
    df_clean = df[essential_cols].copy()
    df_clean.columns = ['country', 'year', 'sector_raw', 'unit', 'waste_value']
    
    # Filter for 2020–2022 and EU27
    df_clean = df_clean[df_clean['year'].isin([2020, 2021, 2022])]
    df_clean = df_clean[df_clean['country'].isin(country_mapping.keys())].copy()
    df_clean['country'] = df_clean['country'].map(country_mapping)
    
    # Sector mapping
    sector_mapping = {
        'Primary production of food - agriculture, fishing and aquaculture': 'Primary Production',
        'Manufacture of food products and beverages': 'Food Manufacturing',
        'Retail and other distribution of food': 'Retail & Distribution',
        'Restaurants and food services': 'Food Services',
        'Total activities by households': 'Households',
        'Total (aggregate changing according to the context)': 'Total'
    }
    print("Unique sector_raw values:", df_clean['sector_raw'].unique())
    df_clean['sector'] = df_clean['sector_raw'].map(sector_mapping).fillna('Unknown')
    print("Unmapped sectors:", df_clean[df_clean['sector'] == 'Unknown']['sector_raw'].unique())
    
    # Convert 'Total' from tonnes to kg per capita
    population_df = population_df[['country', 'year', 'population']].copy()
    df_clean = df_clean.merge(population_df, on=['country', 'year'], how='left')
    df_clean['waste_kg_per_capita'] = np.where(
        df_clean['unit'] == 'Tonne',
        (df_clean['waste_value'] * 1000) / df_clean['population'],  # Convert tonnes to kg
        df_clean['waste_value']
    )
    df_clean = df_clean.drop(['sector_raw', 'unit', 'waste_value', 'population'], axis=1)
    
    # Handle missing values
    print(f"Missing waste_kg_per_capita: {df_clean['waste_kg_per_capita'].isna().sum()}")
    df_clean = df_clean.dropna(subset=['waste_kg_per_capita'])
    
    print(f"Final food waste records: {len(df_clean)}")
    print(f"Countries: {df_clean['country'].nunique()}")
    print(f"Years: {sorted(df_clean['year'].unique())}")
    print(f"Sectors: {df_clean['sector'].unique()}")
    return df_clean

def clean_food_prices_data(file_path):
    print("\n📈 Cleaning Food Prices Dataset...")
    if not os.path.exists(file_path):
        print(f"Error: {file_path} not found. Please check the file path.")
        return pd.DataFrame()
    
    try:
        df = pd.read_csv(file_path)
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return pd.DataFrame()
    
    essential_cols = ['geo', 'TIME_PERIOD', 'OBS_VALUE', 'coicop', 'unit']
    missing_cols = [col for col in essential_cols if col not in df.columns]
    if missing_cols:
        print(f"Error: Missing columns in food prices CSV: {missing_cols}")
        return pd.DataFrame()
    
    df_clean = df[essential_cols].copy()
    df_clean.columns = ['country', 'year', 'hicp_value', 'coicop', 'unit']
    
    # Debug: Print unique coicop and unit values
    print("Unique coicop values:", df_clean['coicop'].unique())
    print("Unique unit values:", df_clean['unit'].unique())
    
    # Filter for CP01 or full name 'Food and non-alcoholic beverages' or fallback to All-items HICP
    if 'CP01' in df_clean['coicop'].unique() or 'Food and non-alcoholic beverages' in df_clean['coicop'].unique():
        print("Using CP01/Food and non-alcoholic beverages")
        df_clean = df_clean[df_clean['coicop'].isin(['CP01', 'Food and non-alcoholic beverages'])]
    else:
        print("Warning: CP01 not found, using All-items HICP as proxy")
        df_clean = df_clean[df_clean['coicop'] == 'All-items HICP']
    
    # Flexible unit filter
    df_clean = df_clean[df_clean['unit'].isin(['Index, 2015=100', 'Annual average index'])]
    df_clean = df_clean.groupby(['country', 'year'])['hicp_value'].mean().reset_index()
    df_clean = df_clean.rename(columns={'hicp_value': 'hicp_index'})
    
    # Filter for 2020–2022 and EU27
    df_clean = df_clean[df_clean['year'].isin([2020, 2021, 2022])]
    df_clean = df_clean[df_clean['country'].isin(country_mapping.keys())].copy()
    df_clean['country'] = df_clean['country'].map(country_mapping)
    
    # Convert year to int to avoid np.int64
    df_clean['year'] = df_clean['year'].astype(int)
    
    print(f"Missing hicp_index: {df_clean['hicp_index'].isna().sum()}")
    df_clean = df_clean.dropna(subset=['hicp_index'])
    
    print(f"Final food prices records: {len(df_clean)}")
    print(f"Countries: {df_clean['country'].nunique()}")
    print(f"Years: {sorted(df_clean['year'].unique())}")
    return df_clean

def clean_gdp_data(file_path, population_df):
    print("\n💰 Cleaning GDP Dataset...")
    if not os.path.exists(file_path):
        print(f"Error: {file_path} not found. Please check the file path.")
        return pd.DataFrame()
    
    try:
        df = pd.read_csv(file_path)
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return pd.DataFrame()
    
    essential_cols = ['geo', 'TIME_PERIOD', 'OBS_VALUE', 'unit']
    missing_cols = [col for col in essential_cols if col not in df.columns]
    if missing_cols:
        print(f"Error: Missing columns in GDP CSV: {missing_cols}")
        return pd.DataFrame()
    
    df_clean = df[essential_cols].copy()
    df_clean.columns = ['country', 'year', 'gdp_value', 'unit']
    
    # Debug: Print unique values
    print("Unique units in GDP.csv:", df_clean['unit'].unique())
    print("Unique countries in GDP.csv:", df_clean['country'].unique())
    print("Unique years in GDP.csv:", df_clean['year'].unique())
    
    # FIXED: Filter for your dataset's unit (per capita, already in euros)
    df_clean = df_clean[df_clean['unit'] == 'Chain linked volumes (2020), euro per capita']
    
    # Since it's already per capita, set gdp_per_capita directly to OBS_VALUE (gdp_value)
    df_clean = df_clean.rename(columns={'gdp_value': 'gdp_per_capita'})
    df_clean = df_clean.drop('unit', axis=1)
    
    # Filter for 2020–2022 and EU27
    df_clean = df_clean[df_clean['year'].isin([2020, 2021, 2022])]
    df_clean = df_clean[df_clean['country'].isin(country_mapping.keys())].copy()
    df_clean['country'] = df_clean['country'].map(country_mapping)
    
    # Convert year to int to avoid np.int64
    df_clean['year'] = df_clean['year'].astype(int)
    
    print(f"Missing gdp_per_capita: {df_clean['gdp_per_capita'].isna().sum()}")
    df_clean = df_clean.dropna(subset=['gdp_per_capita'])
    
    print(f"Final GDP records: {len(df_clean)}")
    print(f"Countries: {df_clean['country'].nunique()}")
    print(f"Years: {sorted(df_clean['year'].unique())}")
    return df_clean

def clean_population_data(file_path):
    print("\n👨‍👩‍👧‍👦 Cleaning Population Dataset...")
    if not os.path.exists(file_path):
        print(f"Error: {file_path} not found. Please check the file path.")
        return pd.DataFrame()
    
    try:
        df = pd.read_csv(file_path)
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return pd.DataFrame()
    
    essential_cols = ['geo', 'TIME_PERIOD', 'OBS_VALUE']
    missing_cols = [col for col in essential_cols if col not in df.columns]
    if missing_cols:
        print(f"Error: Missing columns in Population.csv: {missing_cols}")
        return pd.DataFrame()
    
    df_clean = df[essential_cols].copy()
    df_clean.columns = ['country', 'year', 'population']
    
    # Filter for 2020–2022 and EU27
    df_clean = df_clean[df_clean['year'].isin([2020, 2021, 2022])]
    df_clean = df_clean[df_clean['country'].isin(country_mapping.keys())].copy()
    df_clean['country'] = df_clean['country'].map(country_mapping)
    
    # Convert year to int to avoid np.int64
    df_clean['year'] = df_clean['year'].astype(int)
    
    print(f"Missing population: {df_clean['population'].isna().sum()}")
    df_clean = df_clean.dropna(subset=['population'])
    
    print(f"Final population records: {len(df_clean)}")
    print(f"Countries: {df_clean['country'].nunique()}")
    print(f"Years: {sorted(df_clean['year'].unique())}")
    return df_clean

def merge_all_datasets(food_waste_df, food_prices_df, gdp_df, population_df):
    print("\n🔗 Merging All Datasets...")
    
    # Check if any DataFrame is empty
    if food_waste_df.empty:
        print("⚠️ Food waste DataFrame is empty. Check Waste.csv.")
        return pd.DataFrame()
    if population_df.empty:
        print("⚠️ Population DataFrame is empty. Check Population.csv.")
        return pd.DataFrame()
    if food_prices_df.empty:
        print("⚠️ Food prices DataFrame is empty. Check Food_prices.csv.")
        return pd.DataFrame()
    if gdp_df.empty:
        print("⚠️ GDP DataFrame is empty. Proceeding without GDP (set to NaN in merged).")
        # Optional: Proceed with NaN for GDP, but for now, return empty to avoid issues
        return pd.DataFrame()
    
    # Check year overlaps
    waste_years = set(food_waste_df['year'].unique())
    prices_years = set(food_prices_df['year'].unique())
    gdp_years = set(gdp_df['year'].unique())
    pop_years = set(population_df['year'].unique())
    
    common_years = waste_years.intersection(prices_years, gdp_years, pop_years)
    print(f"Common years across all datasets: {sorted(common_years)}")
    if not common_years:
        print("⚠️ No overlapping years! Check dataset years.")
        return pd.DataFrame()
    
    # Merge datasets
    merged_df = food_waste_df.copy()
    merged_df = merged_df.merge(food_prices_df, on=['country', 'year'], how='left')
    merged_df = merged_df.merge(gdp_df, on=['country', 'year'], how='left')
    merged_df = merged_df.merge(population_df, on=['country', 'year'], how='left')
    
    # Impute missing values with country mean
    for col in ['hicp_index', 'gdp_per_capita']:
        if col in merged_df.columns:
            merged_df[col] = merged_df.groupby('country')[col].transform(lambda x: x.fillna(x.mean()))
    
    # Drop rows with missing critical columns
    df_final = merged_df.dropna(subset=['waste_kg_per_capita', 'population'])
    
    print(f"\n✅ Final merged dataset: {len(df_final)} records")
    print(f"Countries: {df_final['country'].nunique()}")
    print(f"Years: {sorted(df_final['year'].unique())}")
    print(f"Sectors: {df_final['sector'].unique()}")
    return df_final

# Main execution
if __name__ == "__main__":
    # File paths (update if needed)
    waste_file = "Waste.csv"
    population_file = "Population.csv"
    food_prices_file = "Food_prices.csv"  # Or "prc_hicp_aind_page_linear.csv" if not renamed
    gdp_file = "GDP.csv"
    
    # Verify file existence
    for file in [waste_file, population_file, food_prices_file, gdp_file]:
        if not os.path.exists(file):
            print(f"Warning: {file} not found in {os.getcwd()}. Please check the file path.")
    
    # Clean datasets
    population_df = clean_population_data(population_file)
    food_waste_df = clean_food_waste_data(waste_file, population_df)
    food_prices_df = clean_food_prices_data(food_prices_file)
    gdp_df = clean_gdp_data(gdp_file, population_df)
    
    # Merge datasets
    final_df = merge_all_datasets(food_waste_df, food_prices_df, gdp_df, population_df)
    
    if not final_df.empty:
        print("\n📊 Complete Merged Dataset Preview (first 10 rows):")
        print(final_df.head(10).to_string(index=False))
        print(f"\nFull dataset shape: {final_df.shape}")
        final_df.to_csv("merged_dataset.csv", index=False)
        print("\n💾 Merged dataset saved as 'merged_dataset.csv'")
    else:
        print("\n❌ Failed to create merged dataset. Check file paths and data sources.")

Current working directory: /Users/macbookpro/Desktop/Course DSAI books and docs/Semester 1/Research and Methods
Files in directory: ['Module Handbook_506 Research Methods and Scientific Work 2025.pdf', '.DS_Store', 'wastefoodresearch.ipynb', 'Research Methods Week 2.pdf', 'Research Methods Week 1 Jan 2025.pdf', 'Untitled.ipynb', 'Population.csv', 'Waste.csv', 'Food_prices.csv', 'M506_Assessment-brief-2025.pdf', '.ipynb_checkpoints', 'food_waste_analysis.ipynb', 'GDP.csv', 'Research Methods Week 3 Role of Theory and Hypotheses.pdf']

👨‍👩‍👧‍👦 Cleaning Population Dataset...
Missing population: 0
Final population records: 81
Countries: 27
Years: [np.int64(2020), np.int64(2021), np.int64(2022)]
🍽️ Cleaning Food Waste Dataset...
Unique sector_raw values: ['Primary production of food - agriculture, fishing and aquaculture'
 'Manufacture of food products and beverages'
 'Retail and other distribution of food' 'Total activities by households'
 'Restaurants and food services'
 'Total (aggregate 